# 第11回　交絡と疑似相関
## ―― 見せかけの相関を、統制（調整）で見破る

統計学Ⅱ　2026後期　／　北星学園大学　／　小野原 彩香

---

### このノートの使い方

前回（第10回）の交絡を、さらに具体的に解剖する。見せかけの相関を作る交絡を **統制（調整）** で見破る方法と、結論が真逆にひっくり返る **シンプソンのパラドックス** を体験する。▶ を上から押そう。

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
print("準備OK。次のセルへ。")

---
## 1. アイスと水難事故 ―― 共通原因という交絡

「アイスの売上」と「水難事故の件数」を調べると、**強く相関する**。アイスを禁止すれば事故は減る？

もちろん違う。両方を引き起こす **共通の原因＝気温** があるからだ。暑い日はアイスも売れるし、水遊びも増えて事故も増える。アイスと事故の間に直接の因果はない ―― これが **疑似相関（見せかけの相関）**。

まず、その強い相関を確認しよう。

In [ ]:
rng = np.random.default_rng(11)
N = 1000
気温 = rng.uniform(0, 33, N)                                  # 共通原因（交絡）
アイス売上 = 50 + 3.0 * 気温 + rng.normal(0, 8, N)
水難事故 = 2 + 0.30 * 気温 + rng.normal(0, 2, N)
# アイスは水難事故に直接の影響を与えていない（式に入っていない）

r_全体 = np.corrcoef(アイス売上, 水難事故)[0, 1]
print(f"アイス売上 と 水難事故 の相関係数： r = {r_全体:.2f}　← 強い正の相関")
print("だが式を見ると、アイスは事故に一切影響していない。共通原因は『気温』。")

---
## 2. 統制（調整）―― 気温をそろえて見る

交絡を見破る基本は **統制（control）**：交絡変数の影響を **そろえる／差し引いた上で** 関係を見る。

ここでは、アイス売上と水難事故のそれぞれから **気温で説明できる分を回帰で差し引き**、残った部分（残差）どうしの相関＝**偏相関** を見る。気温という共通原因の影響を取り除けば、見せかけの相関は消えるはずだ。

In [ ]:
# 統制：アイス・水難それぞれから『気温で説明できる分』を回帰で引き、残差どうしの相関を見る
def 気温の影響を除いた残差(y):
    傾き, 切片 = np.polyfit(気温, y, 1)      # y を 気温 で回帰
    return y - (傾き * 気温 + 切片)           # 気温で説明できない残り（残差）

残差アイス = 気温の影響を除いた残差(アイス売上)
残差水難 = 気温の影響を除いた残差(水難事故)
r_統制後 = np.corrcoef(残差アイス, 残差水難)[0, 1]

print(f"気温を統制する前の相関　 ： r = {r_全体:+.2f}（強い正）")
print(f"気温を統制した後の偏相関 ： r = {r_統制後:+.2f}  ← ほぼ0。やはり犯人は気温だった")

In [ ]:
# 可視化：色＝気温。同じ色の中（＝同じ気温帯）では右肩上がりになっていない
plt.figure(figsize=(7, 5))
sc = plt.scatter(アイス売上, 水難事故, c=気温, cmap="coolwarm", s=18, alpha=0.7)
plt.colorbar(sc, label="気温（℃）")
plt.xlabel("アイス売上"); plt.ylabel("水難事故")
plt.title(f"全体では右肩上がり(r={r_全体:.2f})だが、同じ色（同じ気温）の中では関係が薄い")
plt.show()

全体としては右肩上がりに見えるが、**同じ色（＝同じ気温）の点だけ**を見ると、その傾きはほとんどない。見えていた相関は、気温という共通原因が作った **疑似相関** だった。

> 💬 **交絡の構造**
> 
> アイス ← 気温 → 水難事故。気温が両方の矢印の根元にある「共通原因」。これを統制（そろえる）すると、アイスと事故の見せかけの関係は消える。第10回のRCTは「ランダム化で交絡を断つ」、今回は「交絡を測って統制で断つ」――どちらも交絡を無力化する道だ。

---
## 3. シンプソンのパラドックス ―― 結論が真逆になる

交絡はときに、**全体と層別で結論を正反対にする**。これが **シンプソンのパラドックス** だ。

架空の例：勉強アプリAとBの「合格率」。**全体で見るとAのほうが高い**。ところが、文系・理系に **分けて見ると、どちらでもBのほうが高い**。そんなことが起こりうる。データで再現しよう。

In [ ]:
# 文系・理系それぞれで、アプリBの方が合格率が高い。だが受験者の偏りで全体は逆転する
data = pd.DataFrame([
    # 学部,   アプリ, 受験, 合格
    ["文系", "A", 200, 140],  # A文系： 70%（Aは受かりやすい文系が多い）
    ["文系", "B",  80,  60],  # B文系： 75%  ← Bが上
    ["理系", "A",  80,  24],  # A理系： 30%
    ["理系", "B", 200,  70],  # B理系： 35%  ← Bが上（Bは受かりにくい理系が多い）
], columns=["学部", "アプリ", "受験", "合格"])
data["合格率"] = data["合格"] / data["受験"]

print("■ 学部ごとに見ると：")
for 学部, g in data.groupby("学部"):
    a = g[g.アプリ=="A"].iloc[0]; b = g[g.アプリ=="B"].iloc[0]
    print(f"  {学部}： A {a.合格率:.0%} vs B {b.合格率:.0%}  → {'B' if b.合格率>a.合格率 else 'A'} が上")

print("\n■ 全体（合算）で見ると：")
for アプリ, g in data.groupby("アプリ"):
    率 = g["合格"].sum() / g["受験"].sum()
    print(f"  アプリ{アプリ}： {率:.0%}（合格{g['合格'].sum()}/受験{g['受験'].sum()}）")
print("\n→ 学部別では両方Bが上なのに、全体ではAが上。結論が逆転！")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# 左：学部別
x = np.arange(2); w = 0.35
for i, (学部, g) in enumerate(data.groupby("学部")):
    率 = [g[g.アプリ==app].iloc[0].合格率 for app in ["A", "B"]]
    axes[0].bar(x + (i-0.5)*w, 率, w, label=学部)
axes[0].set_xticks(x); axes[0].set_xticklabels(["アプリA", "アプリB"])
axes[0].set_title("学部別：どちらもBが上"); axes[0].set_ylabel("合格率"); axes[0].legend()
# 右：全体
全体率 = [data[data.アプリ==app]["合格"].sum()/data[data.アプリ==app]["受験"].sum() for app in ["A","B"]]
axes[1].bar(["アプリA", "アプリB"], 全体率, color=["#e8503a", "#3949ab"])
axes[1].set_title("全体（合算）：Aが上 ← 逆転！"); axes[1].set_ylabel("合格率")
plt.tight_layout(); plt.show()

なぜ逆転するのか。アプリAは **受かりやすい文系** の受験者が多く、アプリBは **受かりにくい理系** が多かった。学部（交絡）の構成比が偏っていたために、全体を合算すると本当の優劣が覆い隠された。

**どちらが正しいのか？** ―― この場合は「学部で統制した（分けて見た）」結論、すなわち **Bが上** が正しい。学部という交絡をそろえずに合算した全体の数字に騙されてはいけない。

---
## 4. 統制すれば何でも解決？ ―― やりすぎの罠（コライダー）

「とにかく手元の変数を全部統制すればいい」――これも誤りだ。

交絡（共通の原因）は統制すべきだが、**合流点（コライダー：2つの原因が共通して引き起こす結果）を統制すると、逆に存在しない相関を作り出してしまう**。何を統制し、何を統制してはいけないかは、変数どうしの因果の向き（どれが原因でどれが結果か）を考えないと決められない。

> 💡 **だから因果ダイアグラム（DAG）**
> 
> 「A→B」のような矢印で変数間の因果の向きを描いた図を **DAG** という。これを描くと、どれが交絡（統制すべき）で、どれがコライダー（統制してはいけない）かが見分けられる。統制は機械的な作業ではなく、因果の仮説に基づく判断だ。

---
## 今日のまとめ

| ポイント | 中身 |
|---|---|
| 疑似相関 | 共通原因（交絡）が作る見せかけの相関（アイス←気温→水難事故） |
| 統制（調整） | 交絡をそろえて見ると、見せかけの相関は消える |
| シンプソンのパラドックス | 全体と層別で結論が逆転。正しいのは交絡で層別したほう |
| やりすぎの罠 | コライダーを統制すると偽の相関を作る。DAGで判断 |

- 強い相関を見たら「共通原因は？」と問い、交絡を統制して確かめる。
- 合算した数字（全体）は、層別すると逆転しうる。

> **課題（Moodle）**：相関事例の交絡候補の指摘と統制の効果（自動採点）＋「この相関は因果か／交絡候補を挙げ、統制したらどうなるか」の批判的記述。詳しくはMoodleの第11回課題を見ること。

> **次回予告**：第12回「情報カスケードと集団意思決定」。ここから今期のクライマックス。個々は合理的なのに、他人の行動を見て自分の情報を捨て、集団そろって間違える ―― 独立が壊れる現場へ。